In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
sales = pd.read_csv("../data/processed/sales_table.csv")
customers = pd.read_csv("../data/processed/customer_table.csv")
delivery = pd.read_csv("../data/processed/delivery_table.csv")
reviews = pd.read_csv("../data/processed/review_table.csv")
payments = pd.read_csv("../data/processed/payment_table.csv")
products = pd.read_csv("../data/processed/product_table.csv")

In [3]:
print("Sales:", sales.shape)
print("Customers:", customers.shape)
print("Delivery:", delivery.shape)
print("Reviews:", reviews.shape)
print("Payments:", payments.shape)
print("Products:", products.shape)

Sales: (112650, 30)
Customers: (99441, 15)
Delivery: (113425, 22)
Reviews: (99992, 18)
Payments: (103887, 15)
Products: (32951, 10)


In [4]:
# Revenue generated from each item
sales["item_revenue"] = sales["price"]

# Freight/shipping cost
sales["freight_cost"] = sales["freight_value"]

# Total value of item including freight
sales["total_item_value"] = (
    sales["item_revenue"] + sales["freight_cost"]
)

display(
    sales[
        [
            "order_id",
            "product_id",
            "price",
            "freight_value",
            "item_revenue",
            "freight_cost",
            "total_item_value"
        ]
    ].head()
)

,order_id,product_id,price,freight_value,item_revenue,freight_cost,total_item_value
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,58.90,13.29,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,239.90,19.93,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,199.00,17.87,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,12.99,12.79,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,199.90,18.14,199.90,18.14,218.04


In [5]:
sales["order_purchase_timestamp"] = pd.to_datetime(
    sales["order_purchase_timestamp"],
    errors="coerce"
)

sales["order_year"] = sales["order_purchase_timestamp"].dt.year
sales["order_month"] = sales["order_purchase_timestamp"].dt.month
sales["order_month_name"] = sales["order_purchase_timestamp"].dt.month_name()

display(
    sales[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_year",
            "order_month",
            "order_month_name"
        ]
    ].head()
)

,order_id,order_purchase_timestamp,order_year,order_month,order_month_name
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13 08:59:02,2017,9,September
1,00018f77f2f0320c557190d7a144bdd3,2017-04-26 10:53:06,2017,4,April
2,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,2018,1,January
3,00024acbcdf0a6daa1e931b038114c75,2018-08-08 10:00:35,2018,8,August
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,2017,2,February


In [6]:
customer_features = (
    sales.groupby("customer_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_spending=("total_item_value", "sum"),
        average_order_value=("total_item_value", "mean"),
        total_items=("product_id", "count")
    )
    .reset_index()
)

display(customer_features.head())

,customer_id,total_orders,total_spending,average_order_value,total_items
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,114.74,1
1,000161a058600d5901f007fab4c27140,1,67.41,67.41,1
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,195.42,1
3,0002414f95344307404f0ace7a26f1d5,1,179.35,179.35,1
4,000379cdec625522490c315e70c7a9fb,1,107.01,107.01,1


In [7]:
customer_features["customer_frequency"] = (
    customer_features["total_orders"]
)

display(customer_features.head())

,customer_id,total_orders,total_spending,average_order_value,total_items,customer_frequency
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,114.74,1,1
1,000161a058600d5901f007fab4c27140,1,67.41,67.41,1,1
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,195.42,1,1
3,0002414f95344307404f0ace7a26f1d5,1,179.35,179.35,1,1
4,000379cdec625522490c315e70c7a9fb,1,107.01,107.01,1,1


In [8]:
customer_last_purchase = (
    sales.groupby("customer_id")["order_purchase_timestamp"]
    .max()
    .reset_index()
)

customer_last_purchase.rename(
    columns={
        "order_purchase_timestamp": "last_purchase_date"
    },
    inplace=True
)

display(customer_last_purchase.head())

,customer_id,last_purchase_date
0,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26
1,000161a058600d5901f007fab4c27140,2017-07-16 09:40:32
2,0001fd6190edaaf884bcaf3d49edf079,2017-02-28 11:06:43
3,0002414f95344307404f0ace7a26f1d5,2017-08-16 13:09:20
4,000379cdec625522490c315e70c7a9fb,2018-04-02 13:42:17


In [9]:
reference_date = sales["order_purchase_timestamp"].max()

customer_last_purchase["customer_recency_days"] = (
    reference_date -
    customer_last_purchase["last_purchase_date"]
).dt.days

display(customer_last_purchase.head())

,customer_id,last_purchase_date,customer_recency_days
0,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,292
1,000161a058600d5901f007fab4c27140,2017-07-16 09:40:32,413
2,0001fd6190edaaf884bcaf3d49edf079,2017-02-28 11:06:43,551
3,0002414f95344307404f0ace7a26f1d5,2017-08-16 13:09:20,382
4,000379cdec625522490c315e70c7a9fb,2018-04-02 13:42:17,153


In [10]:
customer_features = customer_features.merge(
    customer_last_purchase,
    on="customer_id",
    how="left"
)

display(customer_features.head())

,customer_id,total_orders,total_spending,average_order_value,total_items,customer_frequency,last_purchase_date,customer_recency_days
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,114.74,1,1,2017-11-14 16:08:26,292
1,000161a058600d5901f007fab4c27140,1,67.41,67.41,1,1,2017-07-16 09:40:32,413
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,195.42,1,1,2017-02-28 11:06:43,551
3,0002414f95344307404f0ace7a26f1d5,1,179.35,179.35,1,1,2017-08-16 13:09:20,382
4,000379cdec625522490c315e70c7a9fb,1,107.01,107.01,1,1,2018-04-02 13:42:17,153


In [11]:
delivery["order_delivered_customer_date"] = pd.to_datetime(
    delivery["order_delivered_customer_date"],
    errors="coerce"
)

delivery["order_estimated_delivery_date"] = pd.to_datetime(
    delivery["order_estimated_delivery_date"],
    errors="coerce"
)

In [12]:
delivery["delivery_days"] = (
    delivery["order_delivered_customer_date"] -
    pd.to_datetime(delivery["order_purchase_timestamp"], errors="coerce")
).dt.days

display(
    delivery[
        [
            "order_id",
            "delivery_days"
        ]
    ].head()
)

,order_id,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0


In [13]:
delivery["delivery_delay_days"] = (
    delivery["order_delivered_customer_date"] -
    delivery["order_estimated_delivery_date"]
).dt.days

display(
    delivery[
        [
            "order_id",
            "delivery_days",
            "delivery_delay_days"
        ]
    ].head()
)

,order_id,delivery_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,-10.0


In [14]:
delivery["on_time_flag"] = np.where(
    delivery["delivery_delay_days"] <= 0,
    1,
    0
)

display(
    delivery[
        [
            "order_id",
            "delivery_delay_days",
            "on_time_flag"
        ]
    ].head()
)

,order_id,delivery_delay_days,on_time_flag
0,e481f51cbdc54678b7cc49136f2d6af7,-8.0,1
1,53cdb2fc8bc7dce0b6741e2150273451,-6.0,1
2,47770eb9100c2d0c44946d9cf07ec65d,-18.0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,-13.0,1
4,ad21c59c0840e6cb83a9ceb5573f8159,-10.0,1


In [15]:
product_features = (
    sales.groupby("product_id")
    .agg(
        sales_count=("product_id", "count"),
        total_revenue=("item_revenue", "sum"),
        average_price=("price", "mean"),
        average_freight=("freight_cost", "mean")
    )
    .reset_index()
)

display(product_features.head())

,product_id,sales_count,total_revenue,average_price,average_freight
0,00066f42aeeb9f3007548bb9d3f33c38,1,101.65,101.65,18.59
1,00088930e925c41fd95ebfe695fd2655,1,129.90,129.90,13.93
2,0009406fd7479715e4bef61dd91f2462,1,229.00,229.00,13.10
3,000b8f95fcb9e0096488278317764d19,2,117.80,58.90,19.60
4,000d9be29b5207b54e86aa1b1ac54872,1,199.00,199.00,19.27


In [16]:
product_features = product_features.merge(
    products[
        [
            "product_id",
            "product_category_name"
        ]
    ],
    on="product_id",
    how="left"
)

display(product_features.head())

,product_id,sales_count,total_revenue,average_price,average_freight,product_category_name
0,00066f42aeeb9f3007548bb9d3f33c38,1,101.65,101.65,18.59,perfumaria
1,00088930e925c41fd95ebfe695fd2655,1,129.90,129.90,13.93,automotivo
2,0009406fd7479715e4bef61dd91f2462,1,229.00,229.00,13.10,cama_mesa_banho
3,000b8f95fcb9e0096488278317764d19,2,117.80,58.90,19.60,utilidades_domesticas
4,000d9be29b5207b54e86aa1b1ac54872,1,199.00,199.00,19.27,relogios_presentes


In [17]:
seller_features = (
    sales.groupby("seller_id")
    .agg(
        seller_orders=("order_id", "nunique"),
        seller_revenue=("item_revenue", "sum"),
        seller_items=("product_id", "count"),
        average_item_price=("price", "mean"),
        average_freight=("freight_cost", "mean")
    )
    .reset_index()
)

display(seller_features.head())

,seller_id,seller_orders,seller_revenue,seller_items,average_item_price,average_freight
0,0015a82c2db000af6aaaf3ae2ecb0532,3,2685.00,3,895.000000,21.020000
1,001cca7ae9ae17fb1caed9dfb1094831,200,25080.03,239,104.937364,37.046611
2,001e6ad469a905060d959994f1b41e4f,1,250.00,1,250.000000,17.940000
3,002100f778ceb8431b7a1020ff7ab48f,51,1234.50,55,22.445455,14.430182
4,003554e2dce176b5555353e4f3555ac8,1,120.00,1,120.000000,19.380000


In [18]:
review_features = (
    reviews.groupby("order_id")
    .agg(
        average_review_score=("review_score", "mean"),
        review_count=("review_id", "count")
    )
    .reset_index()
)

display(review_features.head())

,order_id,average_review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [19]:
payment_features = (
    payments.groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum"),
        payment_methods=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

display(payment_features.head())

,order_id,total_payment,payment_methods,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3.0
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3.0


In [20]:
customer_features.to_csv(
    "../data/processed/customer_features.csv",
    index=False
)

product_features.to_csv(
    "../data/processed/product_features.csv",
    index=False
)

seller_features.to_csv(
    "../data/processed/seller_features.csv",
    index=False
)

delivery.to_csv(
    "../data/processed/delivery_features.csv",
    index=False
)

review_features.to_csv(
    "../data/processed/review_features.csv",
    index=False
)

payment_features.to_csv(
    "../data/processed/payment_features.csv",
    index=False
)

sales.to_csv(
    "../data/processed/sales_features.csv",
    index=False
)

print("All feature tables saved successfully.")

All feature tables saved successfully.


In [21]:
feature_files = [
    "sales_features.csv",
    "customer_features.csv",
    "product_features.csv",
    "seller_features.csv",
    "delivery_features.csv",
    "review_features.csv",
    "payment_features.csv"
]

for file in feature_files:
    path = "../data/processed/" + file
    
    if os.path.exists(path):
        print("Created:", file)
    else:
        print("Missing:", file)

Created: sales_features.csv
Created: customer_features.csv
Created: product_features.csv
Created: seller_features.csv
Created: delivery_features.csv
Created: review_features.csv
Created: payment_features.csv
